In [6]:
import numpy as np
import numpy.testing as npt
from scipy import stats
from scipy.stats import t,ttest_ind
from scipy.stats import f
from scipy.stats import f_oneway
import statsmodels.api as sm
from statsmodels.regression._prediction import get_prediction
from statsmodels.stats.outliers_influence import OLSInfluence,MLEInfluence
import pandas as pd
from patsy import dmatrices
from numpy.testing import assert_almost_equal, assert_allclose
import matplotlib.pyplot as plt
import seaborn as sns
# some_file.py
import sys
# caution: path[0] is reserved for script path (or '' in REPL)
sys.path.insert(1, r'C:\Users\TODO\Desktop\Abhi\AI\AI\Math\Hands-On\statemodelsStudy')
from olsRegressionAnalysis import dispAnalysisOfVariance, tableDispFormatt,getInvOfProductMat


### See: for vdelails calculations

outliers_influence.py


In [7]:
path  = r"C:\Users\TODO\Desktop\Abhi\AI\AI\allDataSet\bOOK-DataSet\TABLE_3_2_DeliveryTimeData.csv"
df = pd.read_csv(path)
df.columns = ['Obs','DlvrTImeY','NumCaseX1','DstX2']

In [8]:
y, X = dmatrices(
                 formula_like = 'DlvrTImeY ~ NumCaseX1 + DstX2', 
                 data=df,
                 return_type='dataframe'
                 )
res = sm.OLS(y, X).fit()

# print('Significance level: ',f.isf(q = 1-0.35, dfn = res.df_model,dfd = res.df_resid, loc=0, scale=1))


# Method 1: use get_influence() Method

### Get remote point:

1. Calculate H bar = 2 * (res.df_model + 1) / res.nobs
  - if hat_diag > h_bar -> Remote point

2. if di > 3 -> Remote point
  - ei = res.resid
  - di = ei/sqrt(res.mse_resid)
  - ri = standard_resid
  - hii = hat_diag
  - e(i) = ei/(1-hii)
  - ti = student_resid 
  - Ei = square(e(i))
  - COOKS'D = cooks_d


In [9]:
h_bar = 2 * (res.df_model + 1) / res.nobs
infl = res.get_influence()
student = infl.summary_frame()
hat_diag = infl.summary_frame()['hat_diag']

### Method 2: Import OLSInfluence

- Step 1:

MEASURES OF INFLUENCE

Note: leverage of ith obs = sqrt(hii/(1-hii))

1. hii > 2p/n [2 * (res.df_model + 1) / res.nobs]

Concidered levrage Point

2. COOK’S D (No Cutoff, Only Large value of coocks distance will conclude about point of influence)
    Cook’s distance measure is a deletion diagnostic.
    - 2.1). If COOK’S Di  > 0.5, then the ith data point 
          is worthy of further investigation as it may be Leverage/influential.
    - 2.2). If COOK’S Di > 1, then the ith data point is 
          quite likely to be Leverage/influential.
    - 2.3). if COOK’S Di sticks out like a sore thumb from the other Di values, 
          it is almost certainly influential.
    - 2.4). COOK’S Di with F-Stat
          you can compair COOK'S D with Fα,p,n-p & find deleted point
          i would move β(i) to the boundry of apprimanations 100(1-α).
          Genrally for F0.5,p,n-p Di>1 to be influantials i.e. 50%
          'Significance level: ',f.isf(q = 0.05, dfn = res.df_model,dfd = res.df_resid, loc=0, scale=1)
          For ex: D9 = 3.41835, which indicates that deletion of observation 9 
          would move the least-squares estimate to approximately the boundary 
          of a 96% confidence region around . [f.isf(q = 1-0.96, dfn = res.df_model,dfd = res.df_resid, loc=0, scale=1)]
          
          - D22 = 0.45106, and deletion of point 22 will move the estimate of β to approximately 
          the edge of a 35% confidence region. [f.isf(q = 1-0.35, dfn = res.df_model,dfd = res.df_resid, loc=0, scale=1)]

3. if Standardized residuals i.e. 
   - di > 3 -> Leverage/influential.
4. DFBETASj,i where βj claculated for all obs, & βi calculated with deleted ith obs
    A large (in magnitude) value of DFBETASj,i indicates that
    observation i has considerable influence on the jth regression
    coefficient.

    - 4.1). CuttOff
    
       - |DFBETASj,i| > 2/sqrt(obs) then ith Obs warrants examinations.   
       - Note: CuttOff indicate only warrants examinations.

5. DFFITSi = investigate the deletion influence of the ith observation 
              on the predicted or fitted value. Affected by both leverage 
              and prediction error
             (R student * Leaverage)

      - If the data point is an outlier, then R-student will be large in Magnitude
             if the data point has high leverage, hii will be close to unity.
             In either of these cases DFFITSi can be large.
      - 5.1 Cutoff

          - |DFFITSi| > 2 * sqrt(p/n) warrants examinations
          - or          > 2 sqrt(k+2/n-k-2)   

=========================== Leaverage/influence===============================
    hii                     hat_matrix_diag
    
    hat_diag_factor         ??? Factor of diagonal of hat_matrix used in influence

    COOCK's D               cooks_distance          ???    
                            summary_frame()['cooks_d']

    cov_ratio               COVRATIOi

    dffits                  DFFTISi dffits measure for influence of an observation

    dffits_internal         ???
    
    dfbeta                  ???
    
    dfbetas                 DFBETASi
    
    influence               matches the influence measure that gretl reports 
                            u * h / (1 - h) where u are the residuals and h 
                            is the diagonal of the hat_matrix.

========================= MODEL ADEQUACY CHECK ==============================

  - ei                              res.resid
  - di                              ei/sqrt(res.mse_resid)
  - e(i) press                      resid_press 
  - ri                              resid_studentized Studentized residuals using variance from OLS
  - hii                             hat_matrix_diag
  - ti                              resid_studentized_external
                                    R Student/externaly Studentized residuals
                                    Studentized residuals using LOOO variance
  - Ei                              square(e(i))
  - ess_press                       Error sum of squares of PRESS residuals
    
  - resid_studentized_internal      ??? Studentized residuals using variance from OLS    
  - resid_std      ??? estimate of standard deviation of the residuals   
  - resid_var      estimate of variance of the residuals

In [10]:
adequacyCheck = OLSInfluence(res)
#1. Check wd cooks d dist
# Get high influantials point
Di = adequacyCheck.summary_frame()['cooks_d']
print(Di[Di>1].index.values)
# Step 2: Effect On Various parameter
#         Decide Outlier are Leverage or Influence point

# Step 3: Effet on various plot

[8]


Rest Code

TODO or coverd in my example folder